In [98]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
from rdkit.Chem import AllChem
from rdkit import Chem
from sklearn import metrics
import numpy as np
import pandas as pd
import os

In [99]:
def smiles_to_morgan_fp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    """
    Converts a SMILES string to a Morgan fingerprint.
    """
    mol = Chem.MolFromSmiles(smiles)    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

In [100]:
df = pd.read_csv("/home/acomajuncosa/Downloads/activities_CHEMBL1794345.tsv", sep='\t', low_memory=False)
df = df[~df['Smiles'].isna()]
print("No pvalues: " + str([i for i in df['pChEMBL Value'].tolist() if not np.isnan(i)]))
print("All are nM: " + str(set([i for i in df['Standard Units']])))

# Generate pChEMBLs
df['pChEMBL_calculated'] = [-np.log10(i * 1e-09) for i in df['Standard Value']]

# Get actives and inactives
actives = df[df['pChEMBL_calculated'] >= 7]['Smiles'].tolist()
inactives = df[df['pChEMBL_calculated'] < 7]['Smiles'].tolist()
print("Actives: " + str(len(actives)))
print("Inactives: " + str(len(inactives)))

# Fix random seed
np.random.seed(42)

# Choose N actives and 5 * N inactives
N = 7000
selected_actives = np.random.choice(actives, N, replace=False).tolist()
selected_inactives = np.random.choice(inactives, 5 * N, replace=False).tolist()
print("Actives: " + str(len(selected_actives)))
print("Inactives: " + str(len(selected_inactives)))

# Get ECFPs
print("Calculating ECFPs...")
selected_actives = [smiles_to_morgan_fp(i) for i in selected_actives]
selected_inactives = [smiles_to_morgan_fp(i) for i in selected_inactives]

# Create matrices
X = np.array(selected_actives + selected_inactives)
Y = np.array([1]*len(selected_actives) + [0]*len(selected_inactives))
print("Matrix shapes:")
print(X.shape, Y.shape)

No pvalues: []
All are nM: {'nM'}
Actives: 7364
Inactives: 162948
Actives: 7000
Inactives: 35000
Calculating ECFPs...
Matrix shapes:
(42000, 1024) (42000,)


In [101]:
# Split into training and test sets (80% train, 20% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [104]:
# Train a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=8)
clf.fit(X_train, Y_train)
print(f"AUROC TRAIN: {metrics.roc_auc_score(Y_train, clf.predict_proba(X_train)[:,1])}")
print(f"AUROC TEST: {metrics.roc_auc_score(Y_test, clf.predict_proba(X_test)[:,1])}")

AUROC TRAIN: 0.9994666741071428
AUROC TEST: 0.5235219897959182


In [108]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression Classifier
clf_lr = LogisticRegression(random_state=42, max_iter=2500, n_jobs=8)
clf_lr.fit(X_train, Y_train)

# Calculate AUROC for Train and Test sets
train_auroc = metrics.roc_auc_score(Y_train, clf_lr.predict_proba(X_train)[:, 1])
test_auroc = metrics.roc_auc_score(Y_test, clf_lr.predict_proba(X_test)[:, 1])

print(f"AUROC TRAIN: {train_auroc}")
print(f"AUROC TEST: {test_auroc}")

AUROC TRAIN: 0.6523434279336735
AUROC TEST: 0.5349478571428572


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Assuming X_train, Y_train, X_test, Y_test are already defined

# Scale the features for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the SVM with RBF kernel
clf_svm = SVC(kernel='rbf', random_state=42, probability=True)  # Removed n_jobs
clf_svm.fit(X_train_scaled, Y_train)

# Calculate AUROC for Train and Test sets
train_auroc = roc_auc_score(Y_train, clf_svm.predict_proba(X_train_scaled)[:, 1])
test_auroc = roc_auc_score(Y_test, clf_svm.predict_proba(X_test_scaled)[:, 1])

# Output results
print(f"AUROC TRAIN: {train_auroc}")
print(f"AUROC TEST: {test_auroc}")